In [3]:
!pip install faiss-cpu
!pip install transformers
!pip install torch
!pip install -U sentence-transformers

In [4]:
!pip install pandas

In [5]:
#import necessary modules
import json
import pandas as pd
import numpy as np
import torch
import faiss
from transformers import AutoModel, AutoTokenizer

C:\Users\evdok\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import sys
print(sys.executable)
!"{sys.executable}" -m pip install faiss-cpu
!"{sys.executable}" -m pip install torch
!"{sys.executable}" -m pip install transformers

C:\Users\evdok\AppData\Local\Programs\Python\Python311\python.exe


In [25]:
docs = pd.read_csv("IR2025/documents.csv")
print(docs.head())

texts = docs["Text"].astype(str).tolist()  # putting document texts in a list of strings
doc_ids = docs["ID"].tolist()  # putting document ids in a list 

       ID                                               Text
0  193157  Support towards the Europe PMC initiative-Cont...
1  193158  Support to the Vice-Presidents of the ERC Scie...
2  193159  Implementation of activities described in the ...
3  193160  Monitoring Atmospheric Composition and Climate...
4  193161  Pre-Operational Marine Service Continuity in T...


In [155]:
for i in range(3):
    print("Doc ID:", doc_ids[i])
    print("Text:", texts[i][:200], "...\n") 

Doc ID: 193157
Text: Support towards the Europe PMC initiative-Contribution for 2014-2016: "The proposed action will provide continued support to the European Research Council (ERC) in the implementation of its Open Acces ...

Doc ID: 193158
Text: Support to the Vice-Presidents of the ERC Scientific Council 2014: The proposed Action will provide the necessary support to the Vice-Presidents of the European Research Council Scientific Council (ER ...

Doc ID: 193159
Text: Implementation of activities described in the Roadmap to Fusion during Horizon 2020 through a Joint programme of the members of the EUROfusion consortium: A Roadmap to the realization of fusion energy ...



In [26]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

model = SentenceTransformer("all-MiniLM-L6-v2")  # using model all-MiniLM-L6-v2, faster than bert

In [27]:
# making every document into an embedding
# takes about 25 minutes
doc_embeddings = model.encode(
    texts,
    batch_size=32,  # faster with batch processing
    show_progress_bar=True,
    normalize_embeddings=True   # for cosine similarity
)

Batches: 100%|███████████████████████████████████████████████████████████████████████| 573/573 [27:06<00:00,  2.84s/it]


In [19]:
print(doc_embeddings.shape)


(18316, 384)


In [18]:
print(doc_embeddings)

array([[-0.07208095,  0.00099345, -0.03941214, ..., -0.05437931,
        -0.04381926,  0.06455517],
       [-0.10061741, -0.0589565 , -0.04468465, ..., -0.02724987,
        -0.03616611,  0.05243941],
       [-0.11927018,  0.00634582, -0.00936098, ..., -0.05645031,
        -0.03832937,  0.00356104],
       ...,
       [ 0.04070859, -0.04485951,  0.01717314, ...,  0.01224227,
        -0.01049489, -0.02946737],
       [-0.06268685,  0.01314005, -0.06539519, ..., -0.05697434,
        -0.08121118, -0.00444276],
       [ 0.02799379,  0.00792572,  0.03342917, ..., -0.04812807,
         0.0454818 , -0.00225154]], shape=(18316, 384), dtype=float32)

In [81]:
# save the embeddings 
np.save("doc_embeddings.npy", doc_embeddings)
np.save("doc_ids.npy", np.array(doc_ids))

In [28]:
doc_embeddings = np.load("doc_embeddings.npy")  # use the saved embeddings instead of making new ones again
doc_ids = np.load("doc_ids.npy")

dim = doc_embeddings.shape[1]   # embeddings dimention 384

index = faiss.IndexFlatIP(dim)   # making faiss index with inner product (cosine similarity)
index = faiss.IndexIDMap(index)  # pairing embeddings with doc_ids

index.add_with_ids(   # adding the doc_ids
    doc_embeddings.astype("float32"),
    np.array(doc_ids)
)


In [29]:
# saving index
faiss.write_index(index, "faiss_index.bin")

In [34]:
# to make sure index is created correctly
print(index)
print("Index type:", type(index))
print("Number of vectors in index:", index.ntotal)
print("Embedding dimension:", index.d)

<faiss.swigfaiss_avx2.IndexIDMap; proxy of <Swig Object of type 'faiss::IndexIDMapTemplate< faiss::Index > *' at 0x00000195FF423420> >
Index type: <class 'faiss.swigfaiss_avx2.IndexIDMap'>
Number of vectors in index: 18316
Embedding dimension: 384


In [10]:

# using saved index
index = faiss.read_index("faiss_index.bin")


In [11]:
def semantic_search(query, k):  
    query_emb = model.encode(   # making embeddings for the query like for the index
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = index.search(query_emb, k)  # searching k-nearest documents and saving id and similarity score 
    return ids[0], scores[0]

In [1]:
queries = pd.read_csv("IR2025/queries.csv")
print(queries.head())

NameError: name 'pd' is not defined

In [13]:

def queries_run(queries, k):
    results = []
    for _, row in queries.iterrows():
        Qid = row["ID"]      ## from queries file
        Qtext = row["Text"]

        doc_ids, scores = semantic_search(Qtext, k)  # we used IndexFlatIP so the results are already in descending order 

        for rank, (doc_id, score) in enumerate(zip(doc_ids, scores), start=1):
            
            ## format results for trec_eval 
            results.append({    ## put every retrieved document in results list to later put in a file 
                "query_id" : Qid,
                "doc_id" : int(doc_id),
                "rank" : rank,
                "score" : float(score)
                
            })

    return results    

In [14]:
# saving the results to later put in files
results_20 = queries_run(queries, k=20)
print(results_20)
results_30 = queries_run(queries, k=30)
results_50 = queries_run(queries, k=50)

[{'query_id': 'Q01', 'doc_id': 193378, 'rank': 1, 'score': 0.9309734106063843}, {'query_id': 'Q01', 'doc_id': 210137, 'rank': 2, 'score': 0.6186076998710632}, {'query_id': 'Q01', 'doc_id': 198900, 'rank': 3, 'score': 0.6012340784072876}, {'query_id': 'Q01', 'doc_id': 199915, 'rank': 4, 'score': 0.60053950548172}, {'query_id': 'Q01', 'doc_id': 206230, 'rank': 5, 'score': 0.5959801077842712}, {'query_id': 'Q01', 'doc_id': 193836, 'rank': 6, 'score': 0.5952180624008179}, {'query_id': 'Q01', 'doc_id': 193373, 'rank': 7, 'score': 0.5912319421768188}, {'query_id': 'Q01', 'doc_id': 193715, 'rank': 8, 'score': 0.5833245515823364}, {'query_id': 'Q01', 'doc_id': 193375, 'rank': 9, 'score': 0.5811315774917603}, {'query_id': 'Q01', 'doc_id': 206011, 'rank': 10, 'score': 0.5733577013015747}, {'query_id': 'Q01', 'doc_id': 194890, 'rank': 11, 'score': 0.569999098777771}, {'query_id': 'Q01', 'doc_id': 206228, 'rank': 12, 'score': 0.5632987022399902}, {'query_id': 'Q01', 'doc_id': 204144, 'rank': 13, '

In [15]:
## put results in a file for trec eval

def file_results(results, filename, run_name):
    with open(filename, "w") as file:
        for result in results:
            file.write(
                f"{result['query_id']} Q0 {result['doc_id']} "
                f"{result['rank']} {result['score']} {run_name}\n "
            )

In [16]:
file_results(results_20, "run_faiss_k20.txt", "run_faiss_k20")
file_results(results_30, "run_faiss_k30.txt", "run_faiss_k30")
file_results(results_50, "run_faiss_k50.txt", "run_faiss_k50")

In [17]:
# checking file 
with open("run_faiss_k20.txt") as file:
    print(file.read())


Q01 Q0 193378 1 0.9309734106063843 run_faiss_k20
 Q01 Q0 210137 2 0.6186076998710632 run_faiss_k20
 Q01 Q0 198900 3 0.6012340784072876 run_faiss_k20
 Q01 Q0 199915 4 0.60053950548172 run_faiss_k20
 Q01 Q0 206230 5 0.5959801077842712 run_faiss_k20
 Q01 Q0 193836 6 0.5952180624008179 run_faiss_k20
 Q01 Q0 193373 7 0.5912319421768188 run_faiss_k20
 Q01 Q0 193715 8 0.5833245515823364 run_faiss_k20
 Q01 Q0 193375 9 0.5811315774917603 run_faiss_k20
 Q01 Q0 206011 10 0.5733577013015747 run_faiss_k20
 Q01 Q0 194890 11 0.569999098777771 run_faiss_k20
 Q01 Q0 206228 12 0.5632987022399902 run_faiss_k20
 Q01 Q0 204144 13 0.5615097880363464 run_faiss_k20
 Q01 Q0 210133 14 0.5613101124763489 run_faiss_k20
 Q01 Q0 206754 15 0.5582636594772339 run_faiss_k20
 Q01 Q0 212366 16 0.5548574924468994 run_faiss_k20
 Q01 Q0 211346 17 0.5546717047691345 run_faiss_k20
 Q01 Q0 212035 18 0.5535317063331604 run_faiss_k20
 Q01 Q0 193722 19 0.5502331852912903 run_faiss_k20
 Q01 Q0 206232 20 0.5497413873672485 run_fai